In [1]:
import os
import re
from docx import Document
from docx.oxml.ns import nsdecls
from docx.oxml import parse_xml
from docx.shared import Pt

In [2]:
# Define directory and paths
csl_file = "Reference_styles/apa.csl" # Change 'apa.csl' if using a different citation style
tex_file = "Latex_Files/latex_file.tex" # Change 'latex_file.tex' to your LaTeX file name
bib_file = "Latex_Files/bib.bib" # Change 'bib.bib' to your bibliography file name
temp_tex_file = "paper_modified.tex"
output_docx = "output.docx" 
# Load the Word document
input_docx = "output.docx"  # Path to your Word file
final_output_docx = "Final_converted_file.docx"  # Save path for modified Word file

In [3]:
def count_columns_in_latex_table(latex_code):
    lines = latex_code.splitlines()
    max_columns = 0

    for line in lines:
        if "&" in line and "\\" in line:
            column_count = line.count("&") + 1
            max_columns = max(max_columns, column_count)

    return max_columns


def modify_table_format(content):
    # Pattern to capture the entire `tabular` environment content
    pattern = r'(\\begin\{tabular\})\{[^}]*\}(.*?\\hline\s*\n)(.*?\\end\{tabular\})'

    def replacement(match):
        # Extract the parts of the match
        tabular_start = match.group(1)  # `\begin{tabular}{`
        table_header = match.group(2)  # Content before first `\hline`
        table_body = match.group(3)    # Content after first `\hline`

        # Check for nested `tabular` environments
        if re.search(r'\\begin\{tabular\}.*\\end\{tabular\}', table_body, re.DOTALL):
            return match.group(0)  # Return the original content if nested `tabular` is found

        # Calculate the number of columns using `count_columns_in_latex_table`
        num_columns = count_columns_in_latex_table(table_body)

        # Generate a new format with simple centered columns (`|c|`)
        new_format = '|' + 'c|' * num_columns

        # Construct the modified table declaration
        modified_table = f'{tabular_start}{{{new_format}}} \n \hline {table_body}'

        return modified_table

    # Apply the replacement to all `tabular` environments in the content
    modified_content = re.sub(pattern, replacement, content, flags=re.DOTALL)

    return modified_content

<>:34: SyntaxWarning: invalid escape sequence '\h'
<>:34: SyntaxWarning: invalid escape sequence '\h'
/var/folders/zj/t0rd3nr52f598g1xnr2pxv7h0000gn/T/ipykernel_31307/3415674887.py:34: SyntaxWarning: invalid escape sequence '\h'
  modified_table = f'{tabular_start}{{{new_format}}} \n \hline {table_body}'


In [4]:
# Read original LaTeX content
with open(tex_file, 'r', encoding='utf-8') as file:
    content = file.read()

# Modify table formats
modified_content = modify_table_format(content)

# Save modified content to temporary file
with open(temp_tex_file, 'w', encoding='utf-8') as file:
    file.write(modified_content)

In [5]:
!pandoc "{temp_tex_file}" --citeproc --bibliography="{bib_file}" --csl="{csl_file}" -o "{output_docx}"

In [6]:
# Open the document
doc = Document(input_docx)

In [7]:
# Define a function to set borders for a table
def set_table_borders(table):
    for row in table.rows:
        for cell in row.cells:
            # Apply border to each cell
            cell._element.get_or_add_tcPr().append(
                parse_xml(r'<w:tcBorders {}><w:top w:val="single" w:sz="4" w:space="0" w:color="000000"/><w:left w:val="single" w:sz="4" w:space="0" w:color="000000"/><w:bottom w:val="single" w:sz="4" w:space="0" w:color="000000"/><w:right w:val="single" w:sz="4" w:space="0" w:color="000000"/></w:tcBorders>'.format(nsdecls('w')))
            )

# Define a function to set font size for all text in a table
def set_table_font_size(table, font_size):
    for row in table.rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(font_size)

In [8]:
# Loop through all tables in the document, apply borders, and set font size
for table in doc.tables:
    set_table_borders(table)
    set_table_font_size(table, 9)  # Set font size to 8

In [9]:
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn

def apply_times_new_roman_font():
    """
    Applies 'Times New Roman' font style to the entire text of a Word (.docx) document.

    Args:
        docx_path (str): Path to the input .docx file.
        output_path (str): Path to save the modified .docx file.
    """

    # Apply font style to all paragraphs and runs
    for paragraph in doc.paragraphs:
        for run in paragraph.runs:
            run.font.name = 'Times New Roman'
            run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')  # Ensures compatibility for East Asian text
            run.font.size = Pt(12)  # Optionally set font size to 12 pt, a standard size for Times New Roman

    # Apply font style to tables as well
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.font.name = 'Times New Roman'
                        run._element.rPr.rFonts.set(qn('w:eastAsia'), 'Times New Roman')
                        run.font.size = Pt(12)


In [10]:
# Example usage
apply_times_new_roman_font()

In [11]:
# Save the modified document
doc.save(final_output_docx)